In [20]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [21]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [22]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, LongType, BooleanType

bs4_parse_schema = StructType([
    StructField('clean_text', StringType(), True),
    StructField('clean_char_count', LongType(), True),
    StructField('code_block_count', LongType(), True),
    StructField('link_count', LongType(), True),
    StructField('tag_count', LongType(), True),
    StructField('is_xml_like', BooleanType(), True),
])

def _bs4_parse_text(raw_text):
    import re
    import html
    text = '' if raw_text is None else str(raw_text)
    trimmed = text.lstrip()
    is_xml_like = trimmed.startswith('<?xml') or trimmed.startswith('<rss') or trimmed.startswith('<feed') or trimmed.startswith('<xml')

    try:
        from bs4 import BeautifulSoup as _BeautifulSoup
        soup = _BeautifulSoup(text, 'html.parser')
        clean_text = ' '.join(soup.get_text(' ', strip=True).split())
        code_block_count = len(soup.find_all('code')) + len(soup.find_all('pre'))
        link_count = len(soup.find_all('a'))
        tag_count = len(soup.find_all(True))
    except Exception:
        code_block_count = len(re.findall(r'(?is)<\s*(pre|code)\b', text))
        link_count = len(re.findall(r'(?is)<\s*a\b[^>]*href\s*=', text))
        tag_count = len(re.findall(r'(?is)<\s*/?\s*[a-zA-Z][^>]*>', text))
        text_no_tags = re.sub(r'(?is)<[^>]+>', ' ', text)
        clean_text = ' '.join(html.unescape(text_no_tags).split())

    return (clean_text, int(len(clean_text)), int(code_block_count), int(link_count), int(tag_count), bool(is_xml_like))

bs4_parse_text_udf = F.udf(_bs4_parse_text, bs4_parse_schema)

# Read bronze source tables
bronze_metadata = spark.table('ddc_databricks.bronze.pypi_metadata')
bronze_recent = spark.table('ddc_databricks.bronze.pypi_downloads_recent')
bronze_overall = spark.table('ddc_databricks.bronze.pypi_downloads_overall')

In [23]:
# 1) Metadata -> silver.pypi_metadata
metadata_clean = (
    bronze_metadata
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.col('data').cast('string').alias('data_json')
    )
    .withColumn('latest_version', F.get_json_object('data_json', '$.info.version'))
    .withColumn('summary', F.get_json_object('data_json', '$.info.summary'))
    .withColumn('description', F.get_json_object('data_json', '$.info.description'))
    .withColumn('author', F.get_json_object('data_json', '$.info.author'))
    .withColumn('author_email', F.get_json_object('data_json', '$.info.author_email'))
    .withColumn('maintainer', F.get_json_object('data_json', '$.info.maintainer'))
    .withColumn('license', F.get_json_object('data_json', '$.info.license'))
    .withColumn('requires_python', F.get_json_object('data_json', '$.info.requires_python'))
    .withColumn('home_page', F.get_json_object('data_json', '$.info.home_page'))
    .withColumn('project_url', F.get_json_object('data_json', '$.info.project_url'))
    .withColumn('package_url', F.get_json_object('data_json', '$.info.package_url'))
    .withColumn('keywords', F.get_json_object('data_json', '$.info.keywords'))
    .withColumn('summary_char_count', F.length(F.coalesce(F.col('summary'), F.lit(''))))
    .withColumn('description_char_count', F.length(F.coalesce(F.col('description'), F.lit(''))))
    .withColumn('classifiers_count', F.size(F.from_json(F.get_json_object('data_json', '$.info.classifiers'), 'array<string>')))
    .withColumn('requires_dist_count', F.size(F.from_json(F.get_json_object('data_json', '$.info.requires_dist'), 'array<string>')))
    .withColumn('release_count', F.size(F.expr("json_object_keys(get_json_object(data_json, '$.releases'))")))
    .withColumn('project_urls_count', F.size(F.expr("json_object_keys(get_json_object(data_json, '$.info.project_urls'))")))
    .withColumn('has_license', F.when(F.length(F.trim(F.coalesce(F.col('license'), F.lit('')))) > 0, F.lit(True)).otherwise(F.lit(False)))
    .withColumn('has_requires_python', F.when(F.length(F.trim(F.coalesce(F.col('requires_python'), F.lit('')))) > 0, F.lit(True)).otherwise(F.lit(False)))
    .withColumn('description_bs4', bs4_parse_text_udf(F.col('description')))
    .withColumn('description_clean_text', F.col('description_bs4.clean_text'))
    .withColumn('description_clean_char_count', F.col('description_bs4.clean_char_count'))
    .withColumn('description_bs4_code_block_count', F.col('description_bs4.code_block_count'))
    .withColumn('description_bs4_link_count', F.col('description_bs4.link_count'))
    .withColumn('description_html_tag_count', F.col('description_bs4.tag_count'))
    .withColumn('description_xml_like', F.col('description_bs4.is_xml_like'))
    .drop('description_bs4', 'data_json', 'description')
)

metadata_dedup_w = Window.partitionBy('package_name').orderBy(F.col('ingested_at').desc())
metadata_latest = (
    metadata_clean
    .withColumn('rn', F.row_number().over(metadata_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    metadata_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_metadata')
)

print('silver.pypi_metadata created')

silver.pypi_metadata created


In [24]:
# 2) Recent downloads -> silver.pypi_downloads_recent
recent_clean = (
    bronze_recent
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.to_json(F.col('data')).alias('data_json')
    )
    .withColumn('last_day', F.coalesce(F.get_json_object('data_json', '$.data.last_day'), F.get_json_object('data_json', '$.last_day')).cast('long'))
    .withColumn('last_week', F.coalesce(F.get_json_object('data_json', '$.data.last_week'), F.get_json_object('data_json', '$.last_week')).cast('long'))
    .withColumn('last_month', F.coalesce(F.get_json_object('data_json', '$.data.last_month'), F.get_json_object('data_json', '$.last_month')).cast('long'))
    .withColumn('download_velocity_week_over_day', F.when(F.col('last_day') > 0, F.col('last_week') / F.col('last_day')).otherwise(F.lit(None)))
    .withColumn('download_velocity_month_over_week', F.when(F.col('last_week') > 0, F.col('last_month') / F.col('last_week')).otherwise(F.lit(None)))
    .drop('data_json')
)

recent_dedup_w = Window.partitionBy('package_name').orderBy(F.col('ingested_at').desc())
recent_latest = (
    recent_clean
    .withColumn('rn', F.row_number().over(recent_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    recent_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_downloads_recent')
)

print('silver.pypi_downloads_recent created')

silver.pypi_downloads_recent created


In [25]:
# 3) Overall downloads history -> silver.pypi_downloads_overall_daily + silver.pypi_downloads_overall_features
overall_rows_schema = ArrayType(StructType([
    StructField('category', StringType(), True),
    StructField('date', StringType(), True),
    StructField('downloads', StringType(), True)
]))

overall_exploded = (
    bronze_overall
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.to_json(F.col('data')).alias('data_json')
    )
    .withColumn('overall_rows_json', F.coalesce(F.get_json_object('data_json', '$.data'), F.col('data_json')))
    .withColumn('overall_rows', F.from_json(F.col('overall_rows_json'), overall_rows_schema))
    .withColumn('row', F.explode_outer('overall_rows'))
    .select(
        'package_name',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('row.category').alias('category'),
        F.to_date(F.col('row.date')).alias('event_date'),
        F.col('row.downloads').cast('long').alias('downloads')
    )
    .filter(F.col('event_date').isNotNull())
)

overall_dedup_w = Window.partitionBy('package_name', 'category', 'event_date').orderBy(F.col('ingested_at').desc())
overall_daily_latest = (
    overall_exploded
    .withColumn('rn', F.row_number().over(overall_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    overall_daily_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_downloads_overall_daily')
)

overall_features = (
    overall_daily_latest
    .groupBy('package_name')
    .agg(
        F.sum(F.when(F.col('category') == 'without_mirrors', F.col('downloads')).otherwise(F.lit(0))).alias('total_downloads_without_mirrors'),
        F.sum(F.when(F.col('category') == 'with_mirrors', F.col('downloads')).otherwise(F.lit(0))).alias('total_downloads_with_mirrors'),
        F.sum(F.when((F.col('category') == 'without_mirrors') & (F.col('event_date') >= F.date_sub(F.current_date(), 30)), F.col('downloads')).otherwise(F.lit(0))).alias('downloads_30d'),
        F.sum(F.when((F.col('category') == 'without_mirrors') & (F.col('event_date') >= F.date_sub(F.current_date(), 90)), F.col('downloads')).otherwise(F.lit(0))).alias('downloads_90d'),
        F.avg(F.when(F.col('category') == 'without_mirrors', F.col('downloads'))).alias('avg_daily_downloads_without_mirrors'),
        F.max(F.when(F.col('category') == 'without_mirrors', F.col('downloads'))).alias('max_daily_downloads_without_mirrors'),
        F.countDistinct(F.when(F.col('category') == 'without_mirrors', F.col('event_date'))).alias('active_days_without_mirrors'),
        F.max(F.when(F.col('category') == 'without_mirrors', F.col('event_date'))).alias('latest_download_date')
    )
    .withColumn('downloads_momentum_30d_vs_90d', F.when(F.col('downloads_90d') > 0, F.col('downloads_30d') / F.col('downloads_90d')).otherwise(F.lit(None)))
)

(
    overall_features
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_downloads_overall_features')
)

print('silver.pypi_downloads_overall_daily and silver.pypi_downloads_overall_features created')

silver.pypi_downloads_overall_daily and silver.pypi_downloads_overall_features created


In [26]:
# 4) Cross-linked package-level PyPI features -> silver.pypi_package_features
pypi_package_features = (
    metadata_latest.alias('m')
    .join(recent_latest.alias('r'), on='package_name', how='left')
    .join(overall_features.alias('o'), on='package_name', how='left')
    .select(
        F.col('package_name'),
        F.col('m.latest_version'),
        F.col('m.author'),
        F.col('m.license'),
        F.col('m.requires_python'),
        F.col('m.summary_char_count'),
        F.col('m.description_char_count'),
        F.col('m.description_clean_char_count'),
        F.col('m.description_bs4_code_block_count'),
        F.col('m.description_bs4_link_count'),
        F.col('m.description_html_tag_count'),
        F.col('m.description_xml_like'),
        F.col('m.classifiers_count'),
        F.col('m.requires_dist_count'),
        F.col('m.release_count'),
        F.col('m.project_urls_count'),
        F.col('m.has_license'),
        F.col('m.has_requires_python'),
        F.col('r.last_day'),
        F.col('r.last_week'),
        F.col('r.last_month'),
        F.col('r.download_velocity_week_over_day'),
        F.col('r.download_velocity_month_over_week'),
        F.col('o.total_downloads_without_mirrors'),
        F.col('o.total_downloads_with_mirrors'),
        F.col('o.downloads_30d'),
        F.col('o.downloads_90d'),
        F.col('o.avg_daily_downloads_without_mirrors'),
        F.col('o.max_daily_downloads_without_mirrors'),
        F.col('o.active_days_without_mirrors'),
        F.col('o.latest_download_date'),
        F.col('o.downloads_momentum_30d_vs_90d'),
        F.col('m.ingested_at').alias('metadata_ingested_at'),
        F.col('r.ingested_at').alias('recent_ingested_at')
    )
    .withColumn('has_recent_downloads', F.when(F.col('last_month').isNotNull() & (F.col('last_month') > 0), F.lit(True)).otherwise(F.lit(False)))
    .withColumn('has_overall_history', F.when(F.col('total_downloads_without_mirrors').isNotNull(), F.lit(True)).otherwise(F.lit(False)))
)

(
    pypi_package_features
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_package_features')
)

print('silver.pypi_package_features created')

silver.pypi_package_features created


In [27]:
# 5) Basic data quality checks for Silver PyPI outputs
quality_metrics = [
    ('bronze_pypi_metadata_rows', bronze_metadata.count()),
    ('bronze_pypi_downloads_recent_rows', bronze_recent.count()),
    ('bronze_pypi_downloads_overall_rows', bronze_overall.count()),
    ('silver_pypi_metadata_rows', metadata_latest.count()),
    ('silver_pypi_downloads_recent_rows', recent_latest.count()),
    ('silver_pypi_downloads_overall_daily_rows', overall_daily_latest.count()),
    ('silver_pypi_downloads_overall_features_rows', overall_features.count()),
    ('silver_pypi_package_features_rows', pypi_package_features.count()),
    ('silver_pypi_missing_recent_downloads', recent_latest.filter(F.col('last_month').isNull()).count()),
    ('silver_pypi_missing_overall_features', pypi_package_features.filter(F.col('has_overall_history') == F.lit(False)).count()),
    ('silver_pypi_xml_like_descriptions', metadata_latest.filter(F.col('description_xml_like') == F.lit(True)).count()),
    ('silver_pypi_missing_clean_description', metadata_latest.filter(F.col('description_clean_char_count').isNull()).count())
]

quality_df = spark.createDataFrame(quality_metrics, ['metric_name', 'metric_value']).withColumn(
    'checked_at', F.current_timestamp()
)

(
    quality_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.pypi_quality_checks')
)

quality_df.orderBy('metric_name').show(truncate=False)

+-------------------------------------------+------------+--------------------------+
|metric_name                                |metric_value|checked_at                |
+-------------------------------------------+------------+--------------------------+
|bronze_pypi_downloads_overall_rows         |14          |2026-03-26 12:34:50.887118|
|bronze_pypi_downloads_recent_rows          |15          |2026-03-26 12:34:50.887118|
|bronze_pypi_metadata_rows                  |8           |2026-03-26 12:34:50.887118|
|silver_pypi_downloads_overall_daily_rows   |2908        |2026-03-26 12:34:50.887118|
|silver_pypi_downloads_overall_features_rows|8           |2026-03-26 12:34:50.887118|
|silver_pypi_downloads_recent_rows          |8           |2026-03-26 12:34:50.887118|
|silver_pypi_metadata_rows                  |8           |2026-03-26 12:34:50.887118|
|silver_pypi_missing_clean_description      |0           |2026-03-26 12:34:50.887118|
|silver_pypi_missing_overall_features       |0        